In [25]:
import base64
import json
import os
import random
from openai import OpenAI
from pydantic import BaseModel
from typing import List, Dict

定义图像编码函数
- 4o系列：请使用Base64编码将图片转换为字符串格式。
- QwenVL系列：可以直接发送原始图片数据，无需进行编码。

In [26]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

定义QA对

In [27]:
class QA_Pair(BaseModel):
    question: str
    answer: str



##### 类定义：`Cmanager`
`Cmanager`类是一个用于管理网络拓扑图像和相关操作的类。它提供了初始化客户端、设置拓扑图像路径、提取实体、构建问答对以及保存数据到JSON文件的功能。
属性：
- `client`: 用于与API通信的客户端对象。
- `topology_image_path`: 存储拓扑图像文件路径的字符串。
方法：
1. `__init__(self, api_base: str, api_key: str)`：
   - 初始化方法，接收两个参数：`api_base`（API的基础URL）和`api_key`（API的密钥）。
   - 创建客户端实例并存储在`self.client`中。
   - 初始化`topology_image_path`为空字符串。
2. `get_client(self, base_url, api_key)`：
   - 私有方法，用于创建并返回一个`OpenAI`客户端实例。
   - 参数：`base_url`和`api_key`。
3. `set_topology_image(self, image_path: str)`：
   - 设置拓扑图像文件路径。
   - 参数：`image_path`（图像文件的路径）。
4. `extract_entities(self) -> List[str]`：
   - 提取拓扑图像中的网络实体。
   - 检查`topology_image_path`是否已设置，如果没有，则抛出`ValueError`。
   - 将图像编码为base64格式，并构建系统内容字符串。
   - 使用`gpt-4o-mini`模型创建聊天完成请求，提取实体并返回实体列表。
5. `build_qa_pairs(self, entities: List[str], num_pairs: int = 10) -> List[Dict[str, str]]`：
   - 基于提取的实体构建问答对。
   - 参数：`entities`（实体列表），`num_pairs`（要生成的问答对数量，默认为10）。
   - 随机选择实体和问题模板，构建问题并使用`gpt-4o-mini`模型生成答案。
   - 返回包含问题和答案的字典列表。
6. `save_to_json(self, entities: List[str], qa_pairs: List[Dict[str, str]], filename: str)`：
   - 将实体和问答对保存为JSON文件。
   - 参数：`entities`（实体列表），`qa_pairs`（问答对列表），`filename`（要保存的文件名）。

##### 注意事项：
1. **图像路径设置**：在调用`extract_entities`和`build_qa_pairs`方法之前，必须通过`set_topology_image`方法设置拓扑图像路径。
2. **图像编码**：`extract_entities`和`build_qa_pairs`方法中使用了`encode_image`函数对图像进行base64编码，确保该函数已正确实现。
3. **问答对生成**：`build_qa_pairs`方法默认生成10个问答对，可以根据需要调整`num_pairs`参数。
4. **JSON文件保存**：`save_to_json`方法将实体和问答对保存为JSON文件，确保指定的`filename`路径可写。
5. **随机性**：`build_qa_pairs`方法中使用了随机选择实体和问题模板，这可能导致每次生成的问答对不同。

In [28]:
class Cmanager:
    def __init__(self, api_base: str, api_key: str):
        self.client = self.get_client(api_base, api_key)
        self.topology_image_path = ""

    @staticmethod
    def get_client(base_url: str, api_key: str) -> OpenAI:
        """Initialize and return the API client."""
        return OpenAI(base_url=base_url, api_key=api_key)

    def set_topology_image(self, image_path: str):
        """Set the path for the topology image."""
        self.topology_image_path = image_path

    # STEP 1: Extract entities from the topology image
    def extract_entities(self) -> List[str]:
        """Extract named core network elements from the topology image."""
        if not self.topology_image_path:
            raise ValueError("Topology image path is not set.")
        
        base64_image = encode_image(self.topology_image_path)
        content_system = (
            "You are a network topology entity extraction expert specializing in identifying named key network elements. "
            "You will receive an image depicting a network topology. "
            "Your task is to identify **named** core network elements from the image. "
            "Extract a maximum of eight (**ten (10) most crucial named core network elements**) of the core network elements. "
            "If there are fewer than five named core network elements, output all of them. "
            "If the image does not contain at least one named core network element, output an empty list: `[]`. "
            "1. Core Network Elements: "
            "- Essential components like routers, switches, servers, firewalls, etc., that have distinct names in the image. "
            "2. Output Format: "
            "- Output the core network element **names** (not types) as a list of strings without any additional information. "
            "- Example output for an image with named elements: ['R1', 'SW1', 'ServerA'] "
            "- Example output for an image with no named elements: [] "
            "3. Important Notes: "
            "- **Focus solely on named entities. Unnamed elements should be ignored.** "
            "- Extract a **maximum of five** core network element names (full names(including Chinese characters if exists) different names). "
            "- Output the extracted names **exactly as they appear** in the image. "
            "- Ensure the output is a valid list."
        )

        completion = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": content_system},
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "Topology Image"},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            },
                        },
                    ],
                },
            ],
            temperature=0.2,
        )

        response_content = completion.choices[0].message.content.strip()
        try:
            # Parse the response content to extract entities
            list_str = response_content[2:-2]
            entities = [entity.strip("'") for entity in list_str.split(", ")]
            if not all(isinstance(entity, str) for entity in entities):
                raise ValueError("Entities are not in the correct format.")
            return entities
        except Exception as e:
            print(f"Error processing entities: {e}")
            return []

    # STEP 2: Generate QA pairs based on extracted entities
    def build_qa_pairs(self, entities: List[str], num_pairs: int = 5) -> List[Dict[str, str]]:
        """Generate question-answer pairs based on the provided entities."""
        qa_pairs = []
        base64_image = encode_image(self.topology_image_path)

        question_templates = [
            "What is the overall topology structure of the network? Is it star, bus, ring, mesh, or hybrid?",
            "What are the main nodes in this topology diagram? Please list them.",
            "How are the main nodes connected to each other?",
            "What is the central node, and what role does it play?",
            "Please summarize the topology diagram using descriptive language, avoiding hierarchical descriptions, with a length of around 200 words. First, describe the overall architecture, then introduce the main entities and their relationships, followed by a detailed explanation of the central node and its function, and finally, explain the specific purpose of this topology diagram.",
        ]

        max_pairs = min(num_pairs, len(question_templates))

        for i in range(max_pairs):
            question_template = question_templates[i]
            question = question_template

            content_system = (
                "You are a network topology expert assistant. "
                "Your task is to answer questions about a network topology based on a provided image and a list of key node entities. "
                "Instructions: "
                "1. Carefully examine the topology image and corresponding key node entities: Understand the connections and relationships between the key node entities, and use the entity names to identify components. "
                "2. Answer the question accurately and concisely: Provide a clear and direct answer to the question without making assumptions or introducing external information. "
                "3. Output your answer as a string. "
                "Topology Image: "
                "Entities: "
                f"{entities} "
                "Question: "
                f"{question}"
            )

            completion = self.client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": content_system},
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": "Topology Image"},
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/jpeg;base64,{base64_image}"
                                },
                            },
                            {"type": "text", "text": f"{question}"},
                        ],
                    },
                ],
                temperature=0.2,
                max_tokens=4095,
            )
            answer = completion.choices[0].message.content.strip()
            qa_pairs.append({"question": question, "answer": answer})

        return qa_pairs

    def save_to_json(self, entities: List[str], qa_pairs: List[Dict[str, str]], filename: str):
        """Save the extracted entities and QA pairs to a JSON file."""
        data = {"entities": entities, "qa_pairs": qa_pairs}
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4, ensure_ascii=False)


初始化Cmanager实例并设置网络拓扑图像路径

In [29]:
api_base = "https://neudm.zeabur.app/v1"
api_key = "sk-T05m0OqxOgKUjErs8c231e1c02E24573A17977F5E839E91c"
cmanager = Cmanager(api_base=api_base, api_key=api_key)

In [30]:
sub_paths = ["normal"]
data_path = "/Users/leon/Desktop/topo2text/data/test_data/"
res_path = "data/result/test_data/"
for sub in sub_paths:
    topology_image_path = os.path.join(data_path, sub)

    # 列出目录下的所有文件
    for filename in os.listdir(topology_image_path):
        # 构建完整的文件路径
        file_path = os.path.join(topology_image_path, filename)
        if os.path.isfile(file_path):
            cmanager.set_topology_image(file_path)
            try:
                entities = cmanager.extract_entities()
                if entities:
                    print("Extracted network element entities:", entities)
                    qa_pairs = cmanager.build_qa_pairs(entities)
                    print("Built QA pairs:", qa_pairs)

                    # 构建结果保存的目录路径
                    result_dir = os.path.join(res_path, sub)
                    if not os.path.exists(result_dir):
                        os.makedirs(result_dir)

                    # 构建结果文件的完整路径
                    image_name_without_ext = os.path.splitext(filename)[0]
                    output_filename = f"{image_name_without_ext}.json"
                    output_file_path = os.path.join(result_dir, output_filename)

                    cmanager.save_to_json(entities, qa_pairs, output_file_path)
                    print(f"Results saved to {output_file_path}")
            except Exception as e:
                print(f"Error processing file {file_path}: {e}")

Extracted network element entities: ['大数据区域', 'Fleet', 'osquery', 'vpn-server', 'ES']


KeyboardInterrupt: 